# Data Tidying and Transformation: Three Wide Datasets

For this assignment I picked three datasets:
1. NASA JPL Planets (from https://ssd.jpl.nasa.gov/planets/)
2. Box Office Mojo Annual Totals (from https://www.boxofficemojo.com/year/)
3. NYC Population by Borough 1950-2040 (https://catalog.data.gov/dataset/new-york-city-population-by-borough-1950-2040)

Each one came in a wide format, meaning each row had a lot of columns that really should be rows. I used pandas to tidy them up and do some basic analysis.

In [ ]:
import pandas as pd

---
## Dataset 1: NASA Planets

This dataset has one row per planet and a bunch of columns for different physical properties like mass, diameter, gravity, etc. That is a classic wide format. I am going to melt it so each row is just one planet + one property + the value.

In [ ]:
planets = pd.read_csv('planets.csv')
planets

The dataset has 'Unknown' in some cells where data is not available. I will replace those with NaN so pandas can handle them properly.

In [ ]:
planets.replace('Unknown', pd.NA, inplace=True)
planets

Now I will melt it into a tidy format. The Planet column stays as the identifier, and everything else becomes a Property/Value pair.

In [ ]:
planets_tidy = planets.melt(id_vars='Planet', var_name='Property', value_name='Value')
planets_tidy

### Analysis

Now that it is tidy I can easily filter down to one property at a time and compare planets.

In [ ]:
# Pull out just gravity and sort it
gravity = planets_tidy[planets_tidy['Property'] == 'Gravity (m/s^2)'].copy()
gravity['Value'] = pd.to_numeric(gravity['Value'])
gravity = gravity.sort_values('Value', ascending=False).reset_index(drop=True)
print("Planets ranked by surface gravity:")
print(gravity[['Planet', 'Value']].to_string(index=False))

In [ ]:
# Which planet has the most moons?
moons = planets_tidy[planets_tidy['Property'] == 'Number of Moons'].copy()
moons['Value'] = pd.to_numeric(moons['Value'])
moons = moons.sort_values('Value', ascending=False).reset_index(drop=True)
print("Planets ranked by number of moons:")
print(moons[['Planet', 'Value']].to_string(index=False))

In [ ]:
# Hottest and coldest planets
temps = planets_tidy[planets_tidy['Property'] == 'Mean Temperature (C)'].copy()
temps['Value'] = pd.to_numeric(temps['Value'])
print(f"Average temperature across all planets: {temps['Value'].mean():.1f} C")
print(f"Hottest: {temps.loc[temps['Value'].idxmax(), 'Planet']} at {temps['Value'].max()} C")
print(f"Coldest: {temps.loc[temps['Value'].idxmin(), 'Planet']} at {temps['Value'].min()} C")

### Conclusions

Jupiter has the strongest gravity at 23.1 m/s² and Saturn has the most moons at 146. Mercury has the weakest gravity. Venus is the hottest planet at 465 C, which is actually hotter than Mercury even though Mercury is closer to the sun — that is because of Venus's thick atmosphere trapping heat. Neptune is the coldest at -200 C.

---

## Dataset 2: Box Office Mojo (2000-2024)

This dataset has one row per year and columns for the total gross, number of releases, and the top 5 films of that year. The top 5 films are stored as pairs of columns like `#1 Release`, `#1 Gross (USD)`, `#2 Release`, etc. That is very wide. I need to tidy the top films section into a long format.

In [ ]:
box_office = pd.read_csv('box_office.csv')
box_office.head()

I will split this into two tables: one for the yearly totals and one for the top 5 films per year in long format.

In [ ]:
# Yearly totals
yearly = box_office[['Year', 'Total Gross (USD)', 'Releases', 'Average Gross (USD)']].copy()
yearly['Total Gross (USD)'] = pd.to_numeric(yearly['Total Gross (USD)'], errors='coerce')
yearly['Average Gross (USD)'] = pd.to_numeric(yearly['Average Gross (USD)'], errors='coerce')
yearly.head()

In [ ]:
# Top films — loop through each rank and stack them into one table
frames = []
for rank in range(1, 6):
    temp = box_office[['Year', f'#{rank} Release', f'#{rank} Gross (USD)']].copy()
    temp.columns = ['Year', 'Film', 'Gross']
    temp['Rank'] = rank
    frames.append(temp)

top_films = pd.concat(frames, ignore_index=True)
top_films['Gross'] = pd.to_numeric(top_films['Gross'], errors='coerce')
top_films = top_films[['Year', 'Rank', 'Film', 'Gross']].sort_values(['Year', 'Rank']).reset_index(drop=True)
top_films.head(10)

### Analysis

In [ ]:
# Best and worst years
print("Best year at the box office:")
print(yearly.loc[yearly['Total Gross (USD)'].idxmax(), ['Year', 'Total Gross (USD)']])
print()
print("Worst year at the box office:")
print(yearly.loc[yearly['Total Gross (USD)'].idxmin(), ['Year', 'Total Gross (USD)']])

In [ ]:
# Top 10 highest grossing films across all years
print("Top 10 highest grossing films (2000-2024):")
print(top_films.nlargest(10, 'Gross')[['Year', 'Film', 'Gross']].to_string(index=False))

In [ ]:
# Compare average annual gross before and after COVID
pre_covid = yearly[yearly['Year'] < 2020]['Total Gross (USD)'].mean()
post_covid = yearly[yearly['Year'] > 2020]['Total Gross (USD)'].mean()
print(f"Average annual gross before COVID (2000-2019): ${pre_covid:,.0f}")
print(f"Average annual gross after COVID (2021-2024):  ${post_covid:,.0f}")
print(f"Change: {((post_covid - pre_covid) / pre_covid * 100):.1f}%")

### Conclusions

2018 was the best year for the U.S. box office with around $11.9 billion, and 2020 was the worst at just $2.1 billion because of COVID shutting down theaters. Even post-COVID the industry has not fully bounced back — the post-2020 average is still noticeably below the pre-COVID norm. Oppenheimer was the single highest grossing film of the whole dataset.

---

## Dataset 3: NYC Population by Borough (1950-2040)

This dataset has one row per borough and separate columns for each decade: 1950, 1960, all the way to 2040. Plus a matching percentage share column for each year. So it is very wide. I need to melt it into a long format with one row per borough per year.

In [ ]:
nyc = pd.read_csv('nyc_population.csv')
nyc

In [ ]:
# The borough names have some extra whitespace, fix that
nyc['Borough'] = nyc['Borough'].str.strip()

# Grab just the plain year columns (not the share ones)
year_cols = [str(y) for y in range(1950, 2050, 10)]

# Melt into long format
nyc_tidy = nyc[['Borough'] + year_cols].melt(id_vars='Borough', var_name='Year', value_name='Population')
nyc_tidy['Year'] = nyc_tidy['Year'].astype(int)
nyc_tidy['Population'] = pd.to_numeric(nyc_tidy['Population'], errors='coerce')
nyc_tidy = nyc_tidy.sort_values(['Borough', 'Year']).reset_index(drop=True)
nyc_tidy.head(12)

### Analysis

In [ ]:
# Separate the 5 boroughs from the NYC Total row
boroughs = nyc_tidy[nyc_tidy['Borough'] != 'NYC Total']

# Which borough had the most people in 2020?
in_2020 = boroughs[boroughs['Year'] == 2020].sort_values('Population', ascending=False)
print("Borough populations in 2020:")
print(in_2020[['Borough', 'Population']].to_string(index=False))

In [ ]:
# How much has each borough grown from 1950 to the 2040 projection?
pop_1950 = boroughs[boroughs['Year'] == 1950].set_index('Borough')['Population']
pop_2040 = boroughs[boroughs['Year'] == 2040].set_index('Borough')['Population']

growth = pd.DataFrame({
    '1950': pop_1950,
    '2040 (projected)': pop_2040,
    '% Change': ((pop_2040 - pop_1950) / pop_1950 * 100).round(1)
})
print("Population change from 1950 to 2040 projection:")
print(growth.to_string())

In [ ]:
# NYC total population by decade
nyc_total = nyc_tidy[nyc_tidy['Borough'] == 'NYC Total'].sort_values('Year')
print("Total NYC population by decade:")
print(nyc_total[['Year', 'Population']].to_string(index=False))

### Conclusions

Brooklyn is by far the most populated borough with 2.6 million people in 2020. Interestingly Manhattan actually shrinks between 1950 and 2040 (-13.7%), while Staten Island more than triples (+161.6%) in the same period. NYC as a whole hit a low point in 1980 at around 7 million people, then climbed back up steadily. The 2040 projections show the city reaching about 9 million total.